# QTS Project  
## Option Wheel Strategy

This project studies a systematic options wheel strategy implemented on listed American equity options. The framework integrates multi-factor stock selection with option premium harvesting under a margin-based capital structure.



---

**Course:** Quantitative Trading Strategy  
**Group:** Final Project Group PF: J 

**Group Members**

- **Name**: Mingshu Lu   **Student ID**: 12496646
- **Name**: Jackie Zhang   **Student ID**: 12498155
- **Name**: Theo Li          **Student ID**: 12503045
- **Name**: Jessica Xu       **Student ID**: 12503042
- **Name**: Catherine Chen   **Student ID**: 12496600

# Project Overview

This project studies a systematic options wheel strategy implemented on listed American-style equity options written on U.S. large-cap stocks. The framework combines equity selection and option premium harvesting within a unified backtesting structure. We'll use the data from 2013-04-01 to 2024-12-31. The objective is to evaluate whether disciplined short-volatility exposure, when applied to a selected set of liquid underlying equities, can generate attractive risk-adjusted returns over an extended sample period.

The strategy operates through monthly rolling option positions while maintaining a multi-year investment horizon to ensure exposure across different market regimes. Both assignment and premium collection dynamics are explicitly modeled, allowing the portfolio to alternate between cash-secured puts and covered calls. Capital allocation, transaction costs, and execution frictions are incorporated to ensure realistic performance estimation.

The backtest spans more than five years and includes at least five distinct underlying equities, generating sufficient trade frequency for statistical evaluation. Performance is assessed across full-sample and stressed market environments in order to evaluate robustness under varying volatility conditions.

# 0. Strategy Architecture

This session defines the structural architecture of the strategy and clarifies how the equity selection layer interacts with the options execution layer. The objective is to establish a clear mapping between conceptual design and the implementation that follows in code.

The strategy consists of two interacting components:

1. A semiannual equity selection mechanism
2. A monthly options wheel execution mechanism

The equity layer determines the eligible stock universe at each rebalance date, while the option layer generates recurring premium income conditional on that selected universe.

Rebalance frequencies are defined as:

$$
T_{equity} = 6 \text{ months}
$$

$$
T_{option} = 4 \text{ weeks}
$$

The code in subsequent sections will construct these two timing cycles explicitly and implement their interaction within a unified backtesting engine.

# 1. Equity Selection

This session constructs the equity selection mechanism that determines which underlying stocks are eligible for option writing. The objective is to formalize the stock ranking process and produce a time-indexed selected stock set.

The data sources are as follows:
- Daily stock market cap, fundamental, analyst consensus data from WRDS(CRSP/IBES)

The selection rule is based on a composite factor score defined as:

$$
Score_i = 0.5 MV_i + 0.2 Q_i + 0.2 M_i + 0.1 C_i
$$

where $MV_i$, $Q_i$, $M_i$, and $C_i$ denote standardized market capitalization, quality, momentum, and analyst consensus signals.

At each semiannual rebalance date, stocks are ranked cross-sectionally and the top-ranked securities are selected. The selected set is denoted as:

$$
\mathcal{S}_t = \{ i_1, i_2, ..., i_k \}
$$

The code in this session will compute factor signals, perform cross-sectional ranking, and construct the time-series of selected stock pools.

In [ ]:
import pandas as pd
import numpy as np
import wrds

# ==========================================
# 1. Data Ingestion: WRDS Pipeline
# ==========================================
def build_sp500_universe_pipeline(username: str, start_date: str = '2011-01-01', end_date: str = '2024-12-31') -> pd.DataFrame:
    """
    Fetches CRSP pricing (including shares outstanding for Market Cap), 
    Compustat fundamentals, and IBES analyst consensus.
    """
    db = wrds.Connection(wrds_username=username)
    
    # print("Fetching CRSP daily prices and calculating Market Cap...")
    crsp_query = f"""
        SELECT a.date, b.ticker, a.prc, a.shrout
        FROM crsp.dsf AS a
        JOIN crsp.dsenames AS b ON a.permno = b.permno
        JOIN crsp.msp500list AS sp ON a.permno = sp.permno
        WHERE a.date >= '{start_date}' AND a.date <= '{end_date}'
        AND a.date >= sp.start AND (a.date <= sp.ending OR sp.ending IS NULL)
        AND a.date >= b.namedt AND a.date <= b.nameendt
    """
    df_price = db.raw_sql(crsp_query, date_cols=['date'])
    df_price['prc'] = df_price['prc'].abs()
    
    # Calculate Market Cap: Price * Shares Outstanding (shrout is usually in thousands)
    df_price['shrout'] = df_price['shrout'].fillna(0)
    df_price['market_cap'] = df_price['prc'] * df_price['shrout']
    
    # Calculate daily return and 6-month momentum
    df_price = df_price.sort_values(['ticker', 'date'])
    df_price['daily_return'] = df_price.groupby('ticker')['prc'].pct_change(1)
    df_price['mom6'] = df_price.groupby('ticker')['prc'].pct_change(126) 
    
    sp500_tickers = df_price['ticker'].dropna().unique().tolist()
    ticker_str = str(tuple(sp500_tickers)) if len(sp500_tickers) > 1 else f"('{sp500_tickers[0]}')"
        
    # print("Fetching Compustat fundamentals (Quality)...")
    comp_query = f"""
        SELECT datadate AS date, tic AS ticker, ni, seq
        FROM comp.funda
        WHERE tic IN {ticker_str}
        AND datadate >= '{start_date}' AND datadate <= '{end_date}'
        AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
    """
    df_fund = db.raw_sql(comp_query, date_cols=['date'])
    df_fund['quality'] = df_fund['ni'] / df_fund['seq'].replace(0, np.nan)
    df_fund['quality'] = df_fund['quality'].astype(float)
    
    # print("Fetching IBES analyst consensus...")
    ibes_query = f"""
        SELECT statpers AS date, ticker, meanrec AS analyst_consensus
        FROM ibes.recdsum
        WHERE ticker IN {ticker_str}
        AND statpers >= '{start_date}' AND statpers <= '{end_date}'
    """
    df_ibes = db.raw_sql(ibes_query, date_cols=['date'])
    
    db.close()
    
    # print("Merging panel data...")
    df_price.set_index(['date', 'ticker'], inplace=True)
    df_fund.set_index(['date', 'ticker'], inplace=True)
    df_ibes.set_index(['date', 'ticker'], inplace=True)
    
    panel = df_price[['prc', 'daily_return', 'mom6', 'market_cap']].copy()
    panel = panel.join(df_fund[['quality']], how='left')
    panel = panel.join(df_ibes[['analyst_consensus']], how='left')
    
    panel['quality'] = panel.groupby(level='ticker')['quality'].ffill()
    panel['analyst_consensus'] = panel.groupby(level='ticker')['analyst_consensus'].ffill()
    
    panel = panel.reset_index()
    panel = panel[(panel['date'] >= '2013-04-01')]
    
    return panel.dropna()


# ==========================================
# 2. Rebalance Schedule (Semiannual)
# ==========================================
def generate_semiannual_rebalance_dates(panel: pd.DataFrame, start_date: str = '2013-04-01', end_date: str = '2024-12-31') -> list:
    """
    Generates semiannual (every 6 months) rebalance dates.
    Aligns the target dates to the next available valid trading day.
    """
    # print("Generating semiannual rebalance dates...")
    trading_dates = pd.DatetimeIndex(pd.Series(panel['date'].unique()).sort_values())
    
    # Create target dates every 6 months (e.g., Apr 1 and Oct 1)
    target_dates = pd.date_range(start=start_date, end=end_date, freq=pd.DateOffset(months=6))
    
    rebalance_dates = []
    for target in target_dates:
        # Find the first valid trading day on or after the target date
        valid_dates = trading_dates[trading_dates >= target]
        if len(valid_dates) > 0:
            rebalance_dates.append(valid_dates[0].strftime('%Y-%m-%d'))
            
    return rebalance_dates

# ==========================================
# 3. Factor Scoring & Selection
# ==========================================
def generate_universe_table(panel: pd.DataFrame, rebalance_dates: list) -> pd.DataFrame:
    """
    Applies the new 4-factor formula: Score = 0.5*MV + 0.2*Q + 0.2*M + 0.1*C
    """
    # print("Calculating composite scores and ranking equities...")
    rebal_df = panel[panel['date'].isin(pd.to_datetime(rebalance_dates))].copy()
    rebal_df.rename(columns={'date': 'rebalance_date'}, inplace=True)
    
    factors = ['market_cap', 'quality', 'mom6', 'analyst_consensus']
    
    # Step 1: Cross-sectional Z-score standardization
    for factor in factors:
        rebal_df[f'{factor}_z'] = rebal_df.groupby('rebalance_date')[factor].transform(
            lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
        )
        
    # Step 2: Apply the weighted formula
    # NOTE: analyst_consensus is subtracted because a lower IBES score (1) means 'Strong Buy'
    rebal_df['composite_score'] = (
        0.5 * rebal_df['market_cap_z'] + 
        0.2 * rebal_df['quality_z'] + 
        0.2 * rebal_df['mom6_z'] - 
        0.1 * rebal_df['analyst_consensus_z']
    )
    
    # Step 3: Rank and Select Top 20
    rebal_df = rebal_df[rebal_df['prc'] <= 1000.0].copy()
    rebal_df['rank'] = rebal_df.groupby('rebalance_date')['composite_score'].rank(method='first', ascending=False)
    rebal_df['selected_flag'] = (rebal_df['rank'] <= 20).astype(int)
    rebal_df['weight'] = np.where(rebal_df['selected_flag'] == 1, 0.05, 0.0)
    
    # output_cols = ['rebalance_date', 'ticker', 'market_cap', 'quality', 'mom6', 'analyst_consensus', 
    #                'composite_score', 'rank', 'selected_flag', 'weight']
    
    output_cols = ['rebalance_date', 'ticker','rank', 'selected_flag']

    final_universe = rebal_df[output_cols].sort_values(['rebalance_date', 'rank'])
    final_universe = final_universe[final_universe['selected_flag'] == 1].reset_index(drop=True)
    
    return final_universe

if __name__ == "__main__":
    YOUR_USERNAME = 'your_username' 
    
    # Run pipelines
    final_panel = build_sp500_universe_pipeline(username=YOUR_USERNAME)
    semi_annual_dates = generate_semiannual_rebalance_dates(final_panel)
    
    final_universe = generate_universe_table(panel=final_panel, rebalance_dates=semi_annual_dates)

    final_universe.to_parquet('UniverseTable.parquet', index=False)

# 2. Data Loading, Cleaning, and Engineering Pipeline

This section defines the data infrastructure and preprocessing pipeline used to construct the research dataset. The objective is to transform raw equity and option data into a clean, aligned panel suitable for signal construction and backtesting.

The data sources are as follows:

- Daily listed option data from Databento API
- Daily stock price, fundamental, analyst consensus data from WRDS / IBES

All datasets are aligned to a common trading calendar and indexed by date and underlying ticker.

---

## 2.1 Equity Data Loading and Cleaning

This subsection loads daily equity price data from WRDS for the selected stocks.
The output of this step is a cleaned equity panel indexed by date and ticker, which serves as the foundation for factor ranking and portfolio construction.

In [ ]:
if __name__ == "__main__":
    YOUR_USERNAME = 'your_username' 
    
    # Run pipelines
    final_panel = build_sp500_universe_pipeline(username=YOUR_USERNAME)
    
    # Generate Table 2: UnderlyingPrice
    underlying_price = final_panel[['date', 'ticker', 'prc']].rename(columns={'prc': 'close'})
    
    underlying_price.to_parquet('UnderlyingPrice.parquet', index=False)



---

## 2.2 Option Data Loading and Cleaning

This subsection loads daily option data from Databento API. The raw dataset includes option price, strike, expiration date, implied volatility, option delta, and open interest.

Contracts are filtered to approximately 4-week maturity in order to match the monthly rolling design of the wheel strategy. Illiquid contracts are removed based on open interest thresholds. Option records are merged with underlying equity prices to ensure pricing consistency.

The output is an aligned option panel indexed by date, underlying ticker, and contract characteristics.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# 2.2 Option Data Loading and Cleaning (strict version)
# 1) Download needed-only option data from UniverseTable (resumable, skips existing files)
from download_rebalance_needed_options import main as download_needed_options_main

download_needed_options_main()

DATA = Path("data")
DEF_DIR = DATA / "databento_raw" / "definition_needed"
OHL_DIR = DATA / "databento_raw" / "OHLCV-1d_needed"

# 2) Load raw needed chunks
def_files = sorted(DEF_DIR.glob("*.parquet"))
ohl_files = sorted(OHL_DIR.glob("*.parquet"))

definition_needed = pd.concat([pd.read_parquet(p) for p in def_files], ignore_index=True) if def_files else pd.DataFrame(
    columns=["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id", "dte"]
)
ohlcv_needed = pd.concat([pd.read_parquet(p) for p in ohl_files], ignore_index=True) if ohl_files else pd.DataFrame(
    columns=["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id", "mid"]
)

# Normalize dtypes
for c in ["date", "exp_date"]:
    if c in definition_needed.columns:
        definition_needed[c] = pd.to_datetime(definition_needed[c], errors="coerce").dt.normalize()
    if c in ohlcv_needed.columns:
        ohlcv_needed[c] = pd.to_datetime(ohlcv_needed[c], errors="coerce").dt.normalize()

for c in ["strike", "dte"]:
    if c in definition_needed.columns:
        definition_needed[c] = pd.to_numeric(definition_needed[c], errors="coerce")
for c in ["strike", "mid"]:
    if c in ohlcv_needed.columns:
        ohlcv_needed[c] = pd.to_numeric(ohlcv_needed[c], errors="coerce")
for c in ["instrument_id"]:
    if c in definition_needed.columns:
        definition_needed[c] = pd.to_numeric(definition_needed[c], errors="coerce").astype("Int64")
    if c in ohlcv_needed.columns:
        ohlcv_needed[c] = pd.to_numeric(ohlcv_needed[c], errors="coerce").astype("Int64")

# 3) Build OptionTable_rebal (strict: mid=close on rebalance date)
def_base = definition_needed[["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id", "dte"]].dropna().copy()
def_base = def_base[(def_base["exp_date"] > def_base["date"]) & def_base["dte"].between(21, 27, inclusive="both")]
def_base = def_base.sort_values(["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id"]).drop_duplicates(
    ["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id"], keep="last"
)

ohl_key = ohlcv_needed[["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id", "mid"]].dropna().copy()
ohl_key = ohl_key.sort_values(["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id", "mid"]).drop_duplicates(
    ["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id"], keep="last"
)

otr = def_base.merge(
    ohl_key,
    on=["date", "ticker", "cp_flag", "exp_date", "strike", "instrument_id"],
    how="inner",
)
OptionTable_rebal = (
    otr[["date", "ticker", "cp_flag", "exp_date", "strike", "mid", "dte"]]
    .sort_values(["date", "ticker", "cp_flag", "exp_date", "strike", "mid"])
    .drop_duplicates(["date", "ticker", "cp_flag", "exp_date", "strike"], keep="last")
    .reset_index(drop=True)
)

# 4) Build strategy-driven trades and OptionMTMTable (actual traded contracts only)
UnderlyingPrice = pd.read_parquet(DATA / "UnderlyingPrice.parquet")
UniverseTable = pd.read_csv("UniverseTable.csv")

UnderlyingPrice["date"] = pd.to_datetime(UnderlyingPrice["date"], errors="coerce").dt.normalize()
UnderlyingPrice["close"] = pd.to_numeric(UnderlyingPrice["close"], errors="coerce")
UniverseTable["rebalance_date"] = pd.to_datetime(UniverseTable["rebalance_date"], errors="coerce").dt.normalize()

spot = UnderlyingPrice.set_index(["date", "ticker"])["close"]

def get_spot(date_val, ticker_val):
    v = spot.get((date_val, ticker_val), np.nan)
    if pd.notna(v):
        return float(v)
    s = UnderlyingPrice[(UnderlyingPrice["ticker"] == ticker_val) & (UnderlyingPrice["date"] <= date_val)].tail(1)
    return float(s["close"].iloc[0]) if len(s) else np.nan

uni_pairs = UniverseTable[["rebalance_date", "ticker"]].dropna().rename(columns={"rebalance_date": "date"})
uni_pairs = uni_pairs.drop_duplicates().sort_values(["ticker", "date"])

trade_rows = []
for ticker, grp in uni_pairs.groupby("ticker"):
    shares = 0
    for d in grp["date"].sort_values():
        sub = OptionTable_rebal[(OptionTable_rebal["date"] == d) & (OptionTable_rebal["ticker"] == ticker)]
        if sub.empty:
            continue

        s0 = get_spot(d, ticker)
        if pd.isna(s0):
            continue

        if shares == 0:
            puts = sub[sub["cp_flag"] == "P"].copy()
            if puts.empty:
                continue
            choice = puts[puts["strike"] <= s0]
            if choice.empty:
                choice = puts
            choice["dist"] = (s0 - choice["strike"]).abs()
            pick = choice.sort_values(["dist", "dte", "mid"]).iloc[0]
        else:
            calls = sub[sub["cp_flag"] == "C"].copy()
            if calls.empty:
                continue
            choice = calls[calls["strike"] >= s0]
            if choice.empty:
                choice = calls
            choice["dist"] = (choice["strike"] - s0).abs()
            pick = choice.sort_values(["dist", "dte", "mid"]).iloc[0]

        exp = pd.Timestamp(pick["exp_date"]).normalize()
        s_exp = get_spot(exp, ticker)
        if pd.isna(s_exp):
            s_exp = s0

        if pick["cp_flag"] == "P":
            shares = 100 if s_exp < float(pick["strike"]) else 0
        else:
            shares = 0 if s_exp > float(pick["strike"]) else 100

        trade_rows.append(
            {
                "trade_date": d,
                "ticker": ticker,
                "cp_flag": pick["cp_flag"],
                "exp_date": exp,
                "strike": float(pick["strike"]),
            }
        )

TradesTable = pd.DataFrame(trade_rows).drop_duplicates(
    ["trade_date", "ticker", "cp_flag", "exp_date", "strike"], keep="last"
).reset_index(drop=True)

# MTM from OHLCV close only (strict mid=close)
mtm = TradesTable.merge(
    ohl_key.rename(columns={"date": "obs_date"})[["obs_date", "ticker", "cp_flag", "exp_date", "strike", "mid"]],
    on=["ticker", "cp_flag", "exp_date", "strike"],
    how="inner",
)
mtm = mtm[(mtm["obs_date"] >= mtm["trade_date"]) & (mtm["obs_date"] <= mtm["exp_date"])].copy()

OptionMTMTable = (
    mtm.rename(columns={"obs_date": "date"})[["date", "ticker", "cp_flag", "exp_date", "strike", "mid"]]
    .sort_values(["date", "ticker", "cp_flag", "exp_date", "strike", "mid"])
    .drop_duplicates(["date", "ticker", "cp_flag", "exp_date", "strike"], keep="last")
    .reset_index(drop=True)
)

# enforce project date bounds
lb = pd.Timestamp("2013-04-01")
ub = pd.Timestamp("2024-12-31")
OptionTable_rebal = OptionTable_rebal[(OptionTable_rebal["date"] >= lb) & (OptionTable_rebal["date"] <= ub)].reset_index(drop=True)
OptionMTMTable = OptionMTMTable[(OptionMTMTable["date"] >= lb) & (OptionMTMTable["date"] <= ub)].reset_index(drop=True)

# save
OptionTable_rebal.to_parquet(DATA / "OptionTable_rebal.parquet", index=False)
OptionMTMTable.to_parquet(DATA / "OptionMTMTable.parquet", index=False)

print("OptionTable_rebal", OptionTable_rebal.shape)
print("TradesTable", TradesTable.shape)
print("OptionMTMTable", OptionMTMTable.shape)
print("Saved:")
print(" - data/OptionTable_rebal.parquet")
print(" - data/OptionMTMTable.parquet")

---

## 2.3 3-Month T-bill Data Loading and Cleaning

This subsection loads daily 3-month T-bill rate data.

In [ ]:
import pandas_datareader.data as web

def fetch_risk_free_rate(start_date: str = '2013-04-01', end_date: str = '2024-12-31') -> pd.DataFrame:
    """
    Fetches the 3-Month Treasury Bill Secondary Market Rate (DTB3) from FRED.
    """
    # print("Fetching 3-Month T-Bill Risk-Free Rate from FRED...")
    rf = web.DataReader('DTB3', 'fred', start_date, end_date)
    rf = rf.reset_index()
    rf.rename(columns={'DATE': 'date', 'DTB3': 'r_3m_annual_pct'}, inplace=True)
    rf['r_3m_annual_pct'] = rf['r_3m_annual_pct'].ffill()
    return rf

if __name__ == "__main__":
    
    risk_free_df = fetch_risk_free_rate()
    risk_free_df.to_parquet('RiskFree3M.parquet', index=False)

# 3. Option Delta Calculation

In order to construct delta-neutral portfolios and manage option exposures, we must compute the option delta for each contract.

Delta measures the sensitivity of the option price to changes in the underlying asset price. 

$$\Delta = \frac{dV}{dS}$$

where V = option price, S = underlying price

Under the Black–Scholes framework, the delta of European options has a closed-form solution.

## 3.1 Black–Scholes Delta Function

We first implement the analytical delta formula from the Black–Scholes model.

For a call option:

$$\Delta_{call} = N(d1)$$

$$\Delta_{put} = N(d1)-1$$

where

$$d_1=\frac{ln(\frac{S}{K})+(r+\frac{\sigma^2}{2})T}{\sigma\sqrt{T}}$$

and 

- S = underlying price
- K = strike price
- T = time to maturity (years)
- r = risk-free rate
- σ = volatility
- N(⋅) = standard normal CDF

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

/Users/zhangjingyun/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [32]:
def black_scholes_delta(S, K, T, r, sigma, cp):
    if (
        pd.isna(S) or pd.isna(K) or pd.isna(T)
        or pd.isna(sigma) or pd.isna(r)
        or sigma <= 0 or T <= 0
    ):
        return np.nan

    # protect against percentage rate input
    if r > 1:
        r = r / 100

    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))

    if cp == 'C':
        return norm.cdf(d1)

    if cp == 'P':
        return norm.cdf(d1) - 1


- In later sections, we construct delta-neutral portfolios, so accurate delta estimates are required.

## 3.2 Historical Volatility Estimation

The Black–Scholes model requires an estimate of volatility $\sigma$

Instead of using implied volatility (which may not always be available), we estimate realized volatility from historical returns.

We compute:

- Rolling 30-day volatility

- Expanding volatility for early observations

- Cross-sectional fallback if missing

Annualization is done using:

$$\sigma_{annual}=\sigma_{daily}*\sqrt{252}$$


In [8]:
def compute_realized_volatility(price_df, window=30):

    price_df = price_df.sort_values(['ticker','date'])

    price_df['ret'] = price_df.groupby('ticker')['close'].pct_change()

    price_df['sigma'] = (
        price_df.groupby('ticker')['ret']
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
        * np.sqrt(252)
    )

    # expanding volatility for early sample
    expanding_vol = (
        price_df.groupby('ticker')['ret']
        .expanding()
        .std()
        .reset_index(level=0, drop=True)
        * np.sqrt(252)
    )

    price_df['sigma'] = price_df['sigma'].fillna(expanding_vol)

    # final fallback
    price_df['sigma'] = price_df.groupby('date')['sigma'].transform(
        lambda x: x.fillna(x.median())
    )

    return price_df[['date','ticker','sigma']]

- Volatility estimation can be unstable, especially early in the sample.

- This layered approach improves robustness:

| Method | Purpose |
|------|------|
| Rolling volatility | Captures recent market conditions |
| Expanding volatility | Avoids missing early values |
| Cross-sectional median | Final fallback to avoid missing data |

## 3.3 Delta Calculation Pipeline

Next we construct the full delta computation pipeline that performs the following steps:
- Merge option data with underlying prices
- Compute time to maturity
- Estimate volatility
- Apply Black–Scholes delta formula

In [34]:
def compute_option_delta(option_table,
                         underlying_price,
                         riskfree_rate):

    # Merge underlying price
    df = option_table.merge(
        underlying_price,
        on=['date','ticker'],
        how='left'
    )

    df.rename(columns={'close':'S'}, inplace=True)

    # Merge risk-free rate
    df = df.merge(
        riskfree_rate,
        on='date',
        how='left'
    )

    df.rename(columns={'r_3m_annual_pct':'r'}, inplace=True)

    # FIX: convert percent → decimal
    df['r'] = df['r'] / 100

    # Time to maturity
    df['T'] = df['dte'] / 365

    # Estimate volatility
    vol_panel = compute_realized_volatility(underlying_price)

    df = df.merge(
        vol_panel,
        on=['date','ticker'],
        how='left'
    )

    # Compute delta
    df['delta'] = df.apply(
        lambda row: black_scholes_delta(
            S=row['S'],
            K=row['strike'],
            T=row['T'],
            r=row['r'],
            sigma=row['sigma'],
            cp=row['cp_flag']
        ),
        axis=1
    )

    return df

This pipeline converts raw data into a fully enriched option dataset containing:

- underlying price
- volatility estimate
- time to maturity
- delta

This dataset will be used later to:

- construct delta-neutral strategies
- evaluate option exposure
- perform hedging analysis

## 3.4 Run the Pipeline

In [44]:
if __name__ == "__main__":

    OptionTable_rebal = pd.read_parquet("data/OptionTable_rebal.parquet")
    UnderlyingPrice   = pd.read_parquet("data/UnderlyingPrice.parquet")
    RiskFree3M        = pd.read_parquet("data/RiskFree3M.parquet")

    OptionTable_delta = compute_option_delta(
        option_table=OptionTable_rebal,
        underlying_price=UnderlyingPrice,
        riskfree_rate=RiskFree3M
    )

    OptionTable_delta.to_parquet(
        "OptionTable_withDelta.parquet",
        index=False
    )
    

/Users/zhangjingyun/anaconda3/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1216: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/zhangjingyun/anaconda3/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1216: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [45]:
OptionTable_delta.groupby("cp_flag")["delta"].describe()

,count,mean,std,min,25%,50%,75%,max
cp_flag,,,,,,,,
C,19328.0,0.434586,0.319317,2.110119e-47,0.146899,0.402842,0.684813,1.0
P,17455.0,-0.293042,0.276284,-1.000000e+00,-0.475942,-0.225902,-0.039185,0.0


### Interpretation

The results are consistent with theoretical expectations:

Call options

- All deltas lie between 0 and 1

- Mean delta ≈ 0.43, indicating many options are near at-the-money

- Maximum delta reaches 1, corresponding to deep ITM calls

Put options

- All deltas lie between −1 and 0

- Mean delta ≈ −0.29

- Minimum delta reaches −1, corresponding to deep ITM puts

### Conclusion

The distribution of computed deltas satisfies the theoretical bounds implied by the Black–Scholes model, confirming that the delta implementation and data pipeline are functioning correctly.

# 4. Option Wheel Mechanics

This session formalizes the mechanical implementation of the options wheel strategy. The objective is to translate the conceptual wheel process into executable trade rules.

For each selected stock, a 4-week out-of-the-money put is sold. Strike selection is determined by delta targeting, typically under 10-delta and 20-delta regimes. Assignment probability is approximated by:

$$
P(\text{assignment}) \approx |\Delta|
$$

If the put expires in-the-money, the underlying stock is assigned. The strategy then transitions to a covered call position. If the call expires in-the-money, the stock is called away and the process returns to a cash-secured put phase.

The code in this session will simulate option expiration outcomes, handle assignment logic, and implement the state transition between cash, stock holding, and covered call positions.

In [ ]:
# -----------------------------
# Delta-target option selection
# -----------------------------


def _pick_delta_contracts(opt_df: pd.DataFrame, target: float, cp_flag: str, strategy_id: str) -> pd.DataFrame:
    """
    Pick one contract per (date, ticker, cp_flag) closest to the target delta.

    Inputs:
    - opt_df: candidate options already filtered to selected universe
    - target: target delta (e.g., -0.10 for put D10)
    - cp_flag: 'P' for put or 'C' for call
    - strategy_id: output label (e.g., 'D10', 'D20')

    Tie-breaker:
    - If |delta - target| is tied, choose the higher mid price.

    Output columns:
    [strategy_id, rebalance_date, ticker, cp_flag, exp_date, strike, trade_price, delta_used].
    """
    # Keep only the requested option type.
    subset = opt_df[opt_df["cp_flag"] == cp_flag].copy()
    if subset.empty:
        # Return empty sheet with standard schema when no candidates exist.
        return pd.DataFrame(columns=[
            "strategy_id", "rebalance_date", "ticker", "cp_flag",
            "exp_date", "strike", "trade_price", "delta_used"
        ])

    # Compute distance to target delta.
    subset["delta_gap"] = (subset["delta"] - target).abs()

    # Sort by closest delta first, then by higher mid price.
    subset = subset.sort_values(
        ["date", "ticker", "delta_gap", "mid"],
        ascending=[True, True, True, False],
    )

    # Pick top-ranked contract per (date, ticker).
    picked = subset.groupby(["date", "ticker"], as_index=False).first()

    # Standardize output field names.
    picked = picked.rename(columns={
        "date": "rebalance_date",
        "mid": "trade_price",
        "delta": "delta_used",
    })
    picked["strategy_id"] = strategy_id

    return picked[
        [
            "strategy_id", "rebalance_date", "ticker", "cp_flag",
            "exp_date", "strike", "trade_price", "delta_used"
        ]
    ]


# -----------------------------
# Pre-filter to selected universe
# -----------------------------
# Only keep stocks selected in the universe table.
selected = UniverseTable[UniverseTable["selected_flag"] == 1][["rebalance_date", "ticker"]].drop_duplicates()

# Normalize and coerce key fields used in selection.
OptionTable_rebal["cp_flag"] = OptionTable_rebal["cp_flag"].str.upper().str.strip()
OptionTable_rebal["mid"] = pd.to_numeric(OptionTable_rebal["mid"], errors="coerce")
OptionTable_rebal["delta"] = pd.to_numeric(OptionTable_rebal["delta"], errors="coerce")

# Keep only rows for selected tickers on rebalance dates.
filtered_options = OptionTable_rebal.merge(
    selected,
    left_on=["date", "ticker"],
    right_on=["rebalance_date", "ticker"],
    how="inner",
).drop(columns=["rebalance_date"])

# -----------------------------
# Build D10/D20 trade sheets
# -----------------------------
# Put targets: -0.10 and -0.20
put_d10 = _pick_delta_contracts(filtered_options, target=-0.10, cp_flag="P", strategy_id="D10")
put_d20 = _pick_delta_contracts(filtered_options, target=-0.20, cp_flag="P", strategy_id="D20")

# Call targets: +0.10 and +0.20
call_d10 = _pick_delta_contracts(filtered_options, target=0.10, cp_flag="C", strategy_id="D10")
call_d20 = _pick_delta_contracts(filtered_options, target=0.20, cp_flag="C", strategy_id="D20")

# Combine and sort for readability.
PutSheet = pd.concat([put_d10, put_d20], ignore_index=True)
CallSheet = pd.concat([call_d10, call_d20], ignore_index=True)

PutSheet = PutSheet.sort_values(["rebalance_date", "ticker", "strategy_id"]).reset_index(drop=True)
CallSheet = CallSheet.sort_values(["rebalance_date", "ticker", "strategy_id"]).reset_index(drop=True)

# Display outputs for quick inspection.
print("PutSheet")
display(PutSheet)
print("CallSheet")
display(CallSheet)

# 5.Strategy Execution

This session formalizes the execution logic of the strategy.

# 5.1 Capital Allocation and Margin Model

This session defines the capital base and leverage framework under which the strategy operates. The objective is to ensure that position sizing and funding costs are explicitly modeled.

Initial capital is defined as:

$$
C_0 = 10{,}000{,}000
$$

Position sizes are determined subject to margin requirements. If a Reg-T framework is applied, position exposure must satisfy:

$$
Exposure \leq \frac{Capital}{Margin\ Requirement}
$$

If assignment results in stock ownership, borrowing costs are incorporated through a funding rate $r_{borrow}$.

The code in this session will compute margin-adjusted position sizes, update available capital dynamically, and incorporate funding costs into portfolio PnL.

# 5.2 Transaction Costs and Execution Modeling

This session incorporates realistic execution assumptions into the backtest. The objective is to prevent overestimation of strategy performance.

Option commissions are modeled as:

$$
0.3 \text{ USD per contract}
$$

Slippage is modeled as:

$$
P_{exec} = P_{mid}(1 \pm 0.01)
$$

where execution price deviates by 1% from mid-price.

The code in this session will adjust trade prices for commission and slippage and produce both gross and net performance series.

# 5.3 Backtest Engine

This session integrates all previous components into a unified portfolio simulation framework. The objective is to generate a daily time series of portfolio value.

Portfolio value is defined as:

$$
V_t = C_t + \sum Equity_t + \sum Option_t
$$

where $C_t$ denotes cash balance, and positions are marked to market daily.

The code in this session will iterate through time, execute monthly option rolls, update assignment states, adjust capital, and record daily portfolio value.

# 6. Performance Evaluation

This session evaluates the performance of the strategy over the full sample and selected subperiods. The objective is to quantify return, risk, and drawdown characteristics.

Key performance measures include:

- Annualized return  
- Annualized volatility  
- Sharpe ratio  
- Maximum drawdown  
- Value-at-Risk (VaR)  

The code in this session will compute these metrics for both gross and net returns and compare them across volatility regimes.